# ELT Case: Snowflake and Python

Purchase orders say what goods should cost. Supplier invoices say what was actually billed.
This notebook builds a pipeline that puts those two numbers side by side, from five sources in
five different formats, and then answers eight questions about what it shows.

Everything is driven from Python, but the work happens **inside Snowflake** - the code is almost
entirely `cs.execute(...)`.

**How this works.** Fill this notebook in, then commit and push it. The pushed notebook is the
submission - there is nothing else to hand in.

Part 1 builds the pipeline, one step per section. Part 2 answers eight questions about what it
shows.

Comments in the code are enough for Part 1. Each Part 2 question has an empty markdown cell under
it for your own reading of the result - **that is optional**, but some of these numbers have a
story behind them and a sentence saying what you make of one is worth more than the number alone.

Two practical things. The notebook should run **top to bottom on a clean kernel** - if a cell only
works because of something you ran earlier and deleted, it will not work for whoever opens it
next. And **do not commit your Snowflake password**; a secret in a commit stays in the git history
even after you delete the line.


## Part 1 - Build the pipeline

### Step 1 - Connect to Snowflake

Do **not** hard-code your password. This notebook is going into version control, and a secret in
a commit stays in the history even after you delete the line. Set the three environment variables
before launching VS Code, or read them from a file you keep out of the repository.


In [2]:
import os
import glob
import re
import csv
import pathlib
import snowflake.connector
from dotenv import load_dotenv
import snowflake.connector

# connect to Snowflake and create a cursor

conn = snowflake.connector.connect(
    user=os.getenv("USERNAME"),
    password=os.getenv("PASSWORD"),
    account=os.getenv("ACCOUNT_STRING"),
)

cs = conn.cursor()


### Step 2 - Create the Snowflake objects

A warehouse for compute, a database for the case, and two schemas: `STAGE` for the internal
stages and file formats, `CORE` for the tables and views we build from them. Separating the
landing area from the modelled tables keeps it obvious which objects are raw and which are
derived.


In [ ]:
# insert code: create the warehouse, database and the STAGE and CORE schemas


### Step 3 - Load the 41 purchase order files

The files are at line-item level, one per month. Three things to get right:

- **Skip the header row.** `SKIP_HEADER = 1` in the file format, or 41 header rows become data.
- **Do the transformation in the `COPY INTO`.** Select the columns you want and cast them there,
  rather than loading everything as text and fixing it afterwards.
- **Automate the `PUT`.** Iterate with `glob`; stage into `year/month` folders. It makes no
  practical difference at this size, but it is the right habit for data that arrives over time.

Columns dropped as not useful: `Comments` and `InternalComments` are almost entirely NULL,
`LastEditedBy` / `LastEditedWhen` (and their `Right_` duplicates) are audit fields, and
`PackageTypeID` carries a single value.

One data note: a few rows carry the date `2/29/2022`, which does not exist - 2022 was not a leap
year. `TRY_TO_DATE` returns NULL for those rather than failing the load, which is exactly why it
is used here instead of `TO_DATE`.


In [ ]:
# insert code: create the file format, the internal stage, and the CORE.PURCHASES table


In [ ]:
# insert code: PUT every purchases csv into the stage, partitioned by year and month


In [ ]:
# insert code: COPY INTO CORE.PURCHASES, selecting columns and casting types in the same step


### Step 4 - Purchase order totals

Roll the line items up to one row per order. `POAmount` is the sum of
`ReceivedOuters * ExpectedUnitPricePerOuter` - **received**, not ordered. The gap between those
two is the whole point of the case.

`OrderDate` and `SupplierID` come along for the ride because they are constant within an order
and every downstream step needs them.


In [ ]:
# insert code: build CORE.PURCHASE_ORDER_TOTALS with POAmount


### Step 5 - Load and shred the supplier transactions

The XML lands in a `VARIANT` column first, then `LATERAL FLATTEN` turns each `<row>` into a
Snowflake row and `XMLGET` pulls the elements out of it.

Look at what is in the file before deciding what to keep. It is not all invoices: rows with
`TransactionTypeID` 5 are supplier invoices and carry a `PurchaseOrderID`; rows with
`TransactionTypeID` 7 are payments, have no purchase order, and have a zero ex-tax amount. Both
belong in the table - the join in step 6 is what filters to invoices.


In [ ]:
# insert code: file format, stage, raw VARIANT table, and the shredded CORE.SUPPLIER_TRANSACTIONS


### Step 6 - Join orders to invoices

Inner join, so only orders that were invoiced survive. `invoiced_vs_quoted` is
`AmountExcludingTax - POAmount`: positive means the supplier billed more than the value of what
arrived.

Watch the grain. The join is order to invoice, one row each - if you join to `CORE.PURCHASES`
instead of the totals table you get one row per **line item** and every difference is counted
several times over.

The case asks for a materialized view. Snowflake materialized views cannot contain joins, so a
table is the right substitute here, exactly as the instructions allow.


In [ ]:
# insert code: create purchase_orders_and_invoices with the invoiced_vs_quoted field


### Step 7 - Bring the supplier data across from Postgres

The data must not pass through Python. Postgres writes it to a file with `COPY ... TO STDOUT`,
and Snowflake picks the file up from a stage.

The `CREATE TABLE` is generated from the file itself rather than typed by hand - a function that
reads the header and samples the data to pick a type per column. That is reusable; a hand-written
`CREATE TABLE` is not.


In [ ]:
def snowflake_type(values):
    """Pick a Snowflake data type for a column, given a sample of its values."""
    # insert code


def create_table_sql(csv_path, table_name, sample_rows=200):
    """Read a csv header and sample its rows, and return a CREATE TABLE statement."""
    # insert code


In [ ]:
# insert code: export supplier_case from postgres to a file, stage it, and load it


### Step 8 - Weather

The Marketplace subscription is the one thing that cannot be driven from Python - do it once in
the Snowflake web interface (**Data Products → Marketplace → NOAA → Weather & Environment →
Get**). After that, everything is SQL again.

Then three pieces of work:

1. **Load the ZCTA file** so every zip code has coordinates. It is tab delimited with seven
   columns, and the coordinates are the last two - read the header before you write the `COPY`.
2. **Find the nearest station to each supplier zip code.** Snowflake has a built-in `HAVERSINE`
   function, so there is no need to write the trigonometry by hand. Filter the station index to a
   rough bounding box first: comparing eight zip codes against every weather station on earth is
   a lot of arithmetic to throw away.
3. **Build `supplier_zip_code_weather`** - zip code, date, daily high - and join it to the orders.

One trap in the supplier data: at least one `postalpostalcode` is stored with four characters
where the ZCTA file has five. Pad it before you join or that supplier silently disappears.


In [ ]:
# insert code: load the ZCTA zip code / lat / long file


In [ ]:
# insert code: pick the single nearest weather station for each distinct supplier zip code


In [ ]:
# insert code: build supplier_zip_code_weather, then join it to the orders and suppliers


## Part 2 - Questions

Each question has a single numeric answer. Run the query; the query and the number are what is
being marked.

Under each one there is a markdown cell for your own reading of the result. Filling it in is
optional - but try it where you see something worth saying.


### Question 1. Across every purchase order in the data, what is the total value of the goods that were **actually received**? Two decimals.


In [ ]:
# insert code: answer the question above with a single query


*Optional - what do you make of this number?*


### Question 2. Not every row in the supplier transaction XML is an invoice. How many are **not** invoices against a purchase order?


In [ ]:
# insert code: answer the question above with a single query


*Optional - what do you make of this number?*


### Question 3. Across all purchase orders that were invoiced, what is the **total amount billed in excess** of the value of goods received?


In [ ]:
# insert code: answer the question above with a single query


*Optional - what do you make of this number?*


### Question 4. What share of the total received value comes from the **single largest supplier**? As a percentage, two decimals.


In [ ]:
# insert code: answer the question above with a single query


*Optional - what do you make of this number?*


### Question 5. Looking at the monthly total value of goods received, which month saw the **largest increase over the month before it**? Answer as `YYYYMM`.

This one needs a window function: the comparison is between a row and the row before it in time.


In [ ]:
# insert code: answer the question above with a single query


*Optional - what do you make of this number?*


### Question 6. Between the first and last order date, how many **calendar days** went by with no purchase order at all?

You cannot count rows that are not there - the calendar has to be generated first, then the orders anti-joined against it.


In [ ]:
# insert code: answer the question above with a single query


*Optional - what do you make of this number?*


### Question 7. Of the supplier zip codes in `supplier_case`, which one is **furthest north**?


In [ ]:
# insert code: answer the question above with a single query


*Optional - what do you make of this number?*


### Question 8. How many suppliers in `supplier_case` **never appear on a purchase order**?


In [ ]:
# insert code: answer the question above with a single query


*Optional - what do you make of this number?*


## Close the connection


In [ ]:
# insert code: close the cursor and the connection
